In [11]:
from pynq import Overlay
ol = Overlay("ns_filter_bd_wrapper.bit")
ns  = ol.ns_filter_0
dma = ol.axi_dma_0

In [ ]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import time
from pynq import allocate

# ---- Load frame + HR ----
H, W = 1080, 1920
prelr = np.fromfile(
    "/home/xilinx/jupyter_notebooks/pc_filter/TreesAndGrass_1920_1080_30fps_8bit/TEST_PRELR/RANGE_1/frame_014_qp160_prelr.yuv",
    dtype=np.uint8).reshape(H, W)
hr = np.fromfile(
    "/home/xilinx/jupyter_notebooks/pc_filter/TreesAndGrass_1920_1080_30fps_8bit/TEST_HR/frame_014.y",
    dtype=np.uint8).reshape(H, W)

# ---- Ported NS trainer utilities ----
PATCH_SIZE, HALF, FILTER_SIZE = 7, 3, 49

def diamond_23_mask():
    m = np.zeros((7, 7), dtype=bool)
    for r, cs in [(0,[3]),(1,[2,3,4]),(2,[1,2,3,4,5]),(3,[1,2,3,4,5]),
                  (4,[1,2,3,4,5]),(5,[2,3,4]),(6,[3])]:
        m[r, cs] = True
    return m.ravel()

def ns_support_basis(mask):
    mi = np.where(mask)[0]
    pos = {int(i): k for k, i in enumerate(mi)}
    seen, cols = set(), []
    for i in mi:
        i = int(i)
        if i in seen: continue
        j = mask.size - 1 - i
        col = np.zeros(mi.size, dtype=np.float32)
        col[pos[i]] = 1.0
        if j != i: col[pos[j]] = 1.0
        cols.append(col); seen.add(i); seen.add(j)
    return np.stack(cols, axis=1)

mask, basis = diamond_23_mask(), ns_support_basis(diamond_23_mask())

# ---- Train (chunked moment accumulation) ----
print("Training taps...")
t0 = time.time()
sr_f, hr_f = prelr.astype(np.float32)/255.0, hr.astype(np.float32)/255.0
sr_pad = np.pad(sr_f, HALF, mode="symmetric")
Mxx = np.zeros((int(mask.sum()),)*2, dtype=np.float64)
Mxy = np.zeros(int(mask.sum()), dtype=np.float64)
yy = 0.0; N = 0
for r0 in range(0, H, 32):
    r1 = min(r0+32, H)
    slab = sr_pad[r0:r1+2*HALF, :]
    win = sliding_window_view(slab, (PATCH_SIZE, PATCH_SIZE))
    Xm = win[:r1-r0, :W].reshape(-1, FILTER_SIZE)[:, mask].astype(np.float64)
    yv = hr_f[r0:r1].ravel().astype(np.float64)
    Mxx += Xm.T @ Xm; Mxy += Xm.T @ yv; yy += float(yv @ yv); N += yv.size
A = basis.T @ Mxx @ basis + 0.01 * np.eye(basis.shape[1])
b = basis.T @ Mxy
w = basis.sum(axis=0).astype(np.float64)
K = basis.shape[1]
KKT = np.zeros((K+1, K+1))
KKT[:K,:K]=A; KKT[:K,K]=w; KKT[K,:K]=w
rhs = np.concatenate([b, [1.0]])
f_unique = np.linalg.solve(KKT, rhs)[:K]
f_mask = basis @ f_unique

# 7-bit CD
A_K = basis.T @ Mxx @ basis; b_K = basis.T @ Mxy
rep_idx = np.array([int(np.argmax(basis[:,k])) for k in range(K)])
f_uc = f_mask[rep_idx].astype(np.float64)
scale = 63 / max(np.max(np.abs(f_uc)), 1e-12)
f_int = np.clip(np.round(f_uc * scale), -63, 63).astype(np.int64)

def _mse(fint):
    fr = fint.astype(np.float64)/scale
    s = float(w @ fr)
    if abs(s) < 1e-6: return float('inf')
    fs = fr/s
    return float((yy - 2*fs@b_K + fs@A_K@fs)/N)

best = _mse(f_int)
for _ in range(20):
    imp = False
    for k in range(K):
        bd = 0
        for d in (-1, 1):
            tr = f_int.copy(); tr[k] = np.clip(tr[k]+d, -63, 63)
            if tr[k] == f_int[k]: continue
            m = _mse(tr)
            if m < best - 1e-12: best, bd = m, d
        if bd != 0:
            f_int[k] = np.clip(f_int[k]+bd, -63, 63); imp = True
    if not imp: break

ws = int(2*np.sum(f_int[:11]) + f_int[11])
norm_q16 = int(round(65536/ws)) & 0xFFFF if ws != 0 else 0
print(f"  train time: {time.time()-t0:.2f}s")
print(f"  taps: {f_int.tolist()}   weighted_sum: {ws}   norm: {norm_q16}")

# ---- Run HW ----
def reset_dma(dma):
    mm = dma.mmio
    mm.write(0x00, 0x4); mm.write(0x30, 0x4)
    time.sleep(0.01)
    mm.write(0x00, 0x1); mm.write(0x30, 0x1)
    dma.sendchannel._first_transfer = True
    dma.recvchannel._first_transfer = True

print("Running HW...")
for i in range(12):
    ns.write(i*4, int(f_int[i]) & 0x7F)
ns.write(0x30, norm_q16)
ns.write(0x34, W); ns.write(0x38, H)
in_buf  = allocate(shape=(H*W,),     dtype=np.uint32)
out_buf = allocate(shape=((H-6)*W,), dtype=np.uint32)
in_buf[:] = prelr.flatten().astype(np.uint32)
reset_dma(dma)
t0 = time.time()
dma.recvchannel.transfer(out_buf); ns.write(0x3C, 1)
dma.sendchannel.transfer(in_buf)
dma.sendchannel.wait(); dma.recvchannel.wait()
print(f"  HW: {(time.time()-t0)*1000:.1f} ms")
hw_out = (out_buf & 0xFF).reshape((H-6, W)).astype(np.uint8).copy()
del in_buf, out_buf

# ---- Python golden apply ----
print("Applying Python golden...")
t0 = time.time()
mi = np.where(mask)[0]
pairs = []
for k in range(K):
    rows = np.where(basis[:,k] == 1.0)[0]
    pairs.append([int(mi[r]) for r in rows])
src = np.pad(prelr, HALF, mode="symmetric").astype(np.int32)
acc = np.zeros((H, W), dtype=np.int64)
for k, positions in enumerate(pairs):
    ps = np.zeros((H, W), dtype=np.int32)
    for p in positions:
        dr, dc = p//7 - HALF, p%7 - HALF
        ps += src[HALF+dr:HALF+dr+H, HALF+dc:HALF+dc+W]
    acc += int(f_int[k]) * ps
py_full = np.clip((acc * norm_q16 + (1<<15)) >> 16, 0, 255).astype(np.uint8)
print(f"  Python apply: {time.time()-t0:.2f}s")

# ---- ALIGNED comparison ----
start = 3*W + 3
hw_flat = hw_out.flatten()
py_flat = py_full.flatten()
exp_flat = py_flat[start : start + hw_flat.size]
diff = hw_flat.astype(np.int16) - exp_flat.astype(np.int16)
core = diff[:-20]           # drop last flush-tail beats
print(f"\nHW vs Python golden (aligned):")
print(f"  max |diff|:   {int(np.abs(core).max())}")
print(f"  mean |diff|:  {float(np.abs(core).mean()):.4f}")
print(f"  exact match:  {float((core == 0).mean())*100:.2f}%")

# ---- PSNR vs HR (also aligned) ----
hr_flat = hr.flatten()
hr_expected = hr_flat[start : start + hw_flat.size]
def psnr(a, b):
    mse = float(np.mean((a.astype(np.int32) - b.astype(np.int32))**2))
    return float('inf') if mse == 0 else 10*np.log10(255**2/mse)

# Baseline = pre-LR itself vs HR at same positions
prelr_baseline = prelr.flatten()[start : start + hw_flat.size]
print(f"\nPSNR vs HR (over the {hw_flat.size:,} filtered pixels):")
print(f"  pre-LR (no filter):    {psnr(prelr_baseline, hr_expected):.3f} dB")
print(f"  HW NS filter:          {psnr(hw_flat, hr_expected):.3f} dB")
print(f"  Python NS filter:      {psnr(exp_flat, hr_expected):.3f} dB")

Training taps...


In [4]:
H, W = 1080, 1920
prelr = np.fromfile(
    "/home/xilinx/jupyter_notebooks/pc_filter/TreesAndGrass_1920_1080_30fps_8bit/TEST_PRELR/RANGE_1/frame_014_qp160_prelr.yuv",
    dtype=np.uint8).reshape(H, W)

# Identity: center=32, norm=2048 -> DC gain exactly 1
taps_id = [0]*11 + [32]
hw_out = run_hw(ns, dma, prelr, taps_id, 2048)

expected = prelr[3:H-3, 3:W-3]
got      = hw_out[:,   3:W-3]
diff     = got.astype(np.int16) - expected.astype(np.int16)
print(f"Identity check:")
print(f"  max |diff|:  {int(np.abs(diff).max())}")
print(f"  mean |diff|: {float(np.abs(diff).mean()):.4f}")
print(f"  exact match: {float((diff == 0).mean())*100:.2f}%")

Identity check:
  max |diff|:  241
  mean |diff|: 19.8528
  exact match: 15.78%


In [5]:
# Only pair 6 = 10, DC gain 1
taps_p6 = [0,0,0,0,0,0, 10, 0,0,0,0, 0]
norm_p6 = round(65536 / 20)   # = 3277

img = np.zeros((H, W), dtype=np.uint8)
img[500, 500] = 255

hw_out = run_hw(ns, dma, img, taps_p6, norm_p6)
nz_r, nz_c = np.where(hw_out > 0)
print(f"Number of nonzero output pixels: {len(nz_r)}")
print(f"First 10 nonzero locations (in input row/col coords):")
for r, c in zip(nz_r[:10], nz_c[:10]):
    print(f"  hw_out[{r},{c}] = {hw_out[r,c]}   (input row={r+3}, col={c})")

KeyboardInterrupt: 

In [7]:
from pynq import Overlay
import numpy as np
import time
from pynq import allocate

# Fresh reload
ol = Overlay("ns_filter_bd_wrapper.bit")
ns  = ol.ns_filter_0
dma = ol.axi_dma_0

def reset_dma(dma):
    mm = dma.mmio
    mm.write(0x00, 0x4); mm.write(0x30, 0x4)
    time.sleep(0.01)
    mm.write(0x00, 0x1); mm.write(0x30, 0x1)
    dma.sendchannel._first_transfer = True
    dma.recvchannel._first_transfer = True

def run_hw(ns, dma, sr_uint8, taps, norm):
    H, W = sr_uint8.shape
    for i in range(12):
        ns.write(i*4, int(taps[i]) & 0x7F)
    ns.write(0x30, int(norm) & 0xFFFF)
    ns.write(0x34, W)
    ns.write(0x38, H)
    in_buf  = allocate(shape=(H*W,),     dtype=np.uint32)
    out_buf = allocate(shape=((H-6)*W,), dtype=np.uint32)
    in_buf[:] = sr_uint8.flatten().astype(np.uint32)
    reset_dma(dma)
    dma.recvchannel.transfer(out_buf)
    ns.write(0x3C, 1)
    dma.sendchannel.transfer(in_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    out = (out_buf & 0xFF).reshape((H-6, W)).astype(np.uint8).copy()
    del in_buf, out_buf
    return out

H, W = 1080, 1920
prelr = np.fromfile(
    "/home/xilinx/jupyter_notebooks/pc_filter/TreesAndGrass_1920_1080_30fps_8bit/TEST_PRELR/RANGE_1/frame_014_qp160_prelr.yuv",
    dtype=np.uint8).reshape(H, W)

# Identity: center=32, norm=2048
hw_out = run_hw(ns, dma, prelr, [0]*11 + [32], 2048)

# ---- 1. Verify frame regs latched ----
print(f"frame_w readback: {ns.read(0x08)}   frame_h readback: {ns.read(0x0C)}")
print(f"pixel_count:      {ns.read(0x00)}   expected: {(H-6)*W}")
print(f"flags:            0x{ns.read(0x04):x}")

# ---- 2. Compare short strips at 3 different vertical positions ----
for r_hw in [0, 500, 1073]:
    r_input = r_hw + 3     # HW row k = input row k+3
    py_row = prelr[r_input, 100:120]
    hw_row = hw_out[r_hw, 100:120]
    diff = hw_row.astype(np.int16) - py_row.astype(np.int16)
    print(f"\nrow hw={r_hw} (input row {r_input}), cols 100..119:")
    print(f"  prelr:  {py_row.tolist()}")
    print(f"  hw:     {hw_row.tolist()}")
    print(f"  diff:   {diff.tolist()}")

# ---- 3. Global stats + spatial distribution ----
expected = prelr[3:H-3, 3:W-3]
got      = hw_out[:,   3:W-3]
diff     = got.astype(np.int16) - expected.astype(np.int16)
print(f"\nOverall identity check:")
print(f"  max |diff|:  {int(np.abs(diff).max())}")
print(f"  mean |diff|: {float(np.abs(diff).mean()):.4f}")
print(f"  exact match: {float((diff == 0).mean())*100:.2f}%")

# Where are the errors concentrated?
row_err = np.abs(diff).mean(axis=1)
col_err = np.abs(diff).mean(axis=0)
print(f"\nError by row band (mean |diff|):")
for band in range(0, 1074, 100):
    print(f"  rows {band:>4}..{band+99}: {row_err[band:band+100].mean():.2f}")
print(f"\nError by col band (mean |diff|):")
for band in range(0, 1914, 200):
    print(f"  cols {band:>4}..{band+199}: {col_err[band:band+200].mean():.2f}")

frame_w readback: 1920   frame_h readback: 1080
pixel_count:      2062084   expected: 2062080
flags:            0xb

row hw=0 (input row 3), cols 100..119:
  prelr:  [12, 8, 6, 5, 7, 13, 15, 24, 40, 69, 104, 137, 123, 111, 83, 79, 97, 94, 54, 23]
  hw:     [5, 7, 13, 15, 24, 40, 69, 104, 137, 123, 111, 83, 79, 97, 94, 54, 23, 8, 13, 7]
  diff:   [-7, -1, 7, 10, 17, 27, 54, 80, 97, 54, 7, -54, -44, -14, 11, -25, -74, -86, -41, -16]

row hw=500 (input row 503), cols 100..119:
  prelr:  [150, 149, 149, 148, 82, 108, 105, 111, 141, 142, 141, 152, 156, 148, 146, 153, 161, 171, 177, 174]
  hw:     [148, 82, 108, 105, 111, 141, 142, 141, 152, 156, 148, 146, 153, 161, 171, 177, 174, 158, 143, 140]
  diff:   [-2, -67, -41, -43, 29, 33, 37, 30, 11, 14, 7, -6, -3, 13, 25, 24, 13, -13, -34, -34]

row hw=1073 (input row 1076), cols 100..119:
  prelr:  [77, 92, 111, 119, 137, 182, 181, 146, 115, 127, 153, 164, 171, 176, 183, 179, 173, 157, 141, 135]
  hw:     [119, 137, 182, 181, 146, 115, 127, 153,

In [8]:
# HW output stream corresponds to filter centers at raster indices [5763 .. 5763+2062079]
# where 5763 = 3*W + 3
start = 3 * W + 3
n = hw_out.size                     # 2062080
hw_flat    = hw_out.flatten()
prelr_flat = prelr.flatten()
expected_flat = prelr_flat[start : start + n]

diff = hw_flat.astype(np.int16) - expected_flat.astype(np.int16)
print(f"Aligned identity check:")
print(f"  max |diff|:  {int(np.abs(diff).max())}")
print(f"  mean |diff|: {float(np.abs(diff).mean()):.4f}")
print(f"  exact match: {float((diff == 0).mean())*100:.2f}%")

# The very last few beats are flush garbage — exclude them
tail = 20
core = diff[:-tail]
print(f"  excluding last {tail} beats: exact match = {float((core == 0).mean())*100:.2f}%")

Aligned identity check:
  max |diff|:  0
  mean |diff|: 0.0000
  exact match: 100.00%
  excluding last 20 beats: exact match = 100.00%


Training taps...
  train time: 26.50s
  taps: [2, 1, -7, 1, 1, -4, 15, -5, 0, -5, 14, 63]   weighted_sum: 89   norm: 736
Running HW...


KeyboardInterrupt: 